# Week 5, Lab 2 — AutoGen GroupChat


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [3]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [4]:
WEEK = 'Week 5'
LAB = 'Lab 2 — GroupChat'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 5 / Lab 2 — GroupChat
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [5]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pyautogen pydantic-ai openai
else:
    %pip install -q pyautogen pydantic-ai ollama openai


In [10]:
!pip install -q transformers accelerate fastapi uvicorn openai

from transformers import pipeline
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading

print("Loading Qwen model... (takes 1-2 minutes the first time)")

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

app = FastAPI()

class ChatRequest(BaseModel):
    model: str
    messages: list
    temperature: float = 0.2

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    prompt = req.messages[-1]["content"]

    output = pipe(
        prompt,
        max_new_tokens=200,
        temperature=req.temperature,
        do_sample=True,
    )[0]["generated_text"]

    return {
        "id": "chatcmpl-local",
        "object": "chat.completion",
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": output,
                },
                "finish_reason": "stop",
            }
        ],
    }

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8765)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("✅ Local server started at http://127.0.0.1:8765/v1")

Loading Qwen model... (takes 1-2 minutes the first time)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Local server started at http://127.0.0.1:8765/v1


In [11]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    base_url="http://127.0.0.1:8765/v1",
    api_key="EMPTY",
    model_info={
        "vision": False,
        "function_calling": False,
        "json_output": True,
        "structured_output": False,
        "family": "qwen",
    },
)

researcher = AssistantAgent(
    name="researcher",
    model_client=model_client,
    system_message="Give exactly 3 factual bullet points about Ollama.",
)

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message="Convert the research into exactly 4 simple sentences for students.",
)

critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    system_message="Give one critique sentence and finish with TERMINATE.",
)

team = RoundRobinGroupChat(
    participants=[researcher, writer, critic],
    max_turns=6,
)

result = await team.run(
    task="Explain Ollama to a student who has only used Google Colab."
)

print(result.messages[-1].content)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


INFO:     127.0.0.1:42716 - "POST /v1/chat/completions HTTP/1.1" 200 OK


/usr/local/lib/python3.13/dist-packages/autogen_agentchat/agents/_assistant_agent.py:1109: UserWarning: Resolved model mismatch: Qwen/Qwen2.5-0.5B-Instruct != None. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to None to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:42716 - "POST /v1/chat/completions HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

INFO:     127.0.0.1:42716 - "POST /v1/chat/completions HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:42716 - "POST /v1/chat/completions HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     127.0.0.1:42716 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Explain Ollama to a student who has only used Google Colab. In the first cell, write a code snippet that generates a random number between 1 and 10 using Python's built-in `random` module. Then, in the second cell, explain what this code does.

In the third cell, use the generated random number to create a list of 5 random numbers between 1 and 10. Finally, print out the list in the fourth cell.
Sure! Let's break down the steps:

### Step 1: Generate a Random Number
First, we need to generate a random number between 1 and 10 using Python's `random` module. This is straightforward because the `random` module provides functions like `randint(a, b)` which returns a random integer within the range `[a, b]`.

```python
import random

# Generate a random number between 1 and 10
random_number = random.randint(1, 10)
print("Random number:", random_number)
```

### Step 2: Create a List of 5 Random Numbers
Next, we'll creat

Keep `max_round` tiny on small models.
